In [5]:
%cd '/Users/lorenzodelcol/Desktop/RA/REPLICATION DROPBOX (NEW)/Raw data UPDATE/Updated 2026 Series'

/Users/lorenzodelcol/Desktop/RA/REPLICATION DROPBOX (NEW)/Raw data UPDATE/Updated 2026 Series


/opt/anaconda3/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [6]:
import pandas as pd

In [14]:
print("Building Phase C: Auto-filling the GFU Dataset with Smart Mapping...")

# 1. Load the "Skeleton" (The original GFU file)
# Note: make sure the filename here matches exactly what you have in your folder
gfu_skeleton = pd.read_excel('cc_globalfactor_1992M72020M5.xlsx', sheet_name='raw data')
target_columns = gfu_skeleton.columns.tolist()

# 2. Load the three Phase B dataframes
mkt_df  = pd.read_excel('PHASE_B_MKT_monthly_df.xlsx', index_col='Date', parse_dates=True)
ex_df   = pd.read_excel('PHASE_B_EX_monthly_df.xlsx', index_col='Date', parse_dates=True)
bond_df = pd.read_excel('PHASE_B_10YR_monthly_df.xlsx', index_col='Date', parse_dates=True)

# This removes any invisible spaces at the beginning or end of the column names
mkt_df.columns = [str(c).strip() for c in mkt_df.columns]
ex_df.columns = [str(c).strip() for c in ex_df.columns]
bond_df.columns = [str(c).strip() for c in bond_df.columns]

# 3. Create the New Timeline
start_date = '1992-07-01'
max_date = max(mkt_df.index.max(), ex_df.index.max(), bond_df.index.max())
new_dates = pd.date_range(start=start_date, end=max_date, freq='MS')

new_gfu = pd.DataFrame(index=new_dates, columns=target_columns)
new_gfu = new_gfu.drop(columns=['Date'])

# ==========================================
# IL TRADUTTORE (The Mapping Dictionary)
# ==========================================
aliases = {
    'Great Britain': ['UK', 'GB', 'Great Britain', 'United Kingdom'],
    'Czech Republic': ['Czech Rep', 'Czech', 'CZ'],
    'New Zeland': ['NZ', 'New Zealand'],
    'New Zealand': ['NZ'],
    'HK': ['Hong Kong', 'HK'],
    'Philippines': ['Philippine', 'Philippines', 'Phillipines'] # Added the double-L typo!
}

def get_phase_b_column(df, country):
    """Cerca la colonna usando il nome originale, poi gli alias, poi le varianti US/USA"""
    # 1. Prova il match esatto
    if f"SD {country}" in df.columns:
        return f"SD {country}"
    
    # 2. Prova il traduttore scorrendo LA LISTA degli alias
    if country in aliases:
        if isinstance(aliases[country], list):
            for possible_name in aliases[country]:
                if f"SD {possible_name}" in df.columns:
                    return f"SD {possible_name}"
        else:
            if f"SD {aliases[country]}" in df.columns:
                return f"SD {aliases[country]}"
        
    # 3. Gestione specifica US vs USA
    if country == 'US' and 'SD USA' in df.columns:
        return 'SD USA'
    if country == 'USA' and 'SD US' in df.columns:
        return 'SD US'
        
    return None # Serie non trovata

# ==========================================
# IL MOTORE DI INSERIMENTO (This was missing!)
# ==========================================
for col in new_gfu.columns:
    parts = col.rsplit('_', 1)
    
    if len(parts) == 2:
        country = parts[0].replace('_', ' ') 
        var_type = parts[1].lower()
        
        # Capisce in quale dataframe guardare
        target_df = None
        if var_type.startswith('st') or var_type == 's': 
            target_df = mkt_df
        elif var_type.startswith('ex') or var_type == 'e': 
            target_df = ex_df
        elif var_type.startswith('by') or var_type == 'b': 
            target_df = bond_df
            
        if target_df is not None:
            # Usa la funzione intelligente per trovare il nome esatto della colonna
            sd_col_name = get_phase_b_column(target_df, country)
            
            if sd_col_name:
                new_gfu[col] = target_df[sd_col_name]
            else:
                print(f" Attention: Haven't ound data for '{country}' ({var_type})")

# ==========================================
# 6. Final Polish
# ==========================================
new_gfu.index.name = 'Date'
new_gfu.reset_index(inplace=True)
new_gfu['Date'] = pd.to_datetime(new_gfu['Date']).dt.strftime('%d/%m/%Y')

# 7. Save the final output
output_name = 'PHASE_C_Updated_GFU_Dataset.xlsx'
new_gfu.to_excel(output_name, index=False)

print(f"\n Success! The dataset is mapped and saved as '{output_name}'")

Building Phase C: Auto-filling the GFU Dataset with Smart Mapping...


/opt/anaconda3/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


 Attention: Haven't ound data for 'Argentina' (by)
 Attention: Haven't ound data for 'Pakistan' (by)
 Attention: Haven't ound data for 'Philippines' (by)


/var/folders/_7/z6fx6jdx1h90fmhzn2dtnxfr0000gn/T/ipykernel_23802/2999730849.py:94: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  new_gfu.reset_index(inplace=True)



 Success! The dataset is mapped and saved as 'PHASE_C_Updated_GFU_Dataset.xlsx'
